In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
from rdkit import Chem
from rdkit.Chem import PeriodicTable
from rdkit.Chem import AllChem
from rdkit.Chem import rdFingerprintGenerator as rdf
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import keras
from collections import defaultdict
from rdkit.Chem.rdPartialCharges import ComputeGasteigerCharges
from sklearn.model_selection import train_test_split
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFreeSASA, rdqueries
from supporting_functions_2_commented import *
import matplotlib.pyplot as plt
import ast


In [2]:
df = pd.read_csv('FULLY_cleaned_ONE_HOT_ENCODED.csv', index_col=0)
df['reactivity_map'] = df['reactivity_map'].apply(ast.literal_eval)
df.head(5)

,substrate_smiles_mapped,product_smiles_mapped,temperature_degC,reactivity_map,BOHR_CC1(C)OB([B:1]2OC(C)(C)C(C)(C)O2)OC1(C)C,BOHR_CC1(C)O[BH:1]OC1(C)C,BOHR_CC1(C)OB(O[B:1]2OC(C)(C)C(C)(C)O2)OC1(C)C,BOHR_c1ccc2c(c1)O[BH:1]O2,BOHR_CC[Si](CC)(CC)[B:1]1OC(C)(C)C(C)(C)O1,BOHR_C1COB([B:1]2OCCO2)O1,...,OTHER_OC(C)(C)C(O)(C)C,OTHER_O(C=1C=C(OC)C=C(OC)C1)C,OTHER_[K].O=S(=O)(O)OOS(=O)(=O)O,OTHER_O1BOC(C)(C)C1(C)C,OTHER_O(C)CCOC,OTHER_ClC(Cl)C(Cl)Cl,OTHER_[Na].O=S(=O)(O)C1=CC=C(C=C1)C,OTHER_[F-][B+3]([F-])([F-])[F-].CC[N+](CC)(CC)CC,reaction_id,new
0,[Br:1][c:2]1[cH:3][cH:4][c:5]([Br:6])[cH:7][cH...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,{4: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,lit_pub_bo_222,False
1,[Br:1][c:2]1[cH:3][cH:4][c:5]2[nH:6][cH:7][cH:...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{4: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,lit_pub_bo_27,False
2,[Br:1][c:2]1[cH:3][cH:4][cH:5][c:6]([Br:7])[cH...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,{4: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,lit_pub_bo_206,False
3,[Br:1][c:2]1[cH:3][cH:4][cH:5][cH:6][c:7]1[Br:8],[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{5: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,lit_pub_bo_207,False
4,[Br:1][c:2]1[cH:3][cH:4][cH:5][s:6]1,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,{5: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,lit_pub_bo_216,False


In [3]:
#df.hist(bins = 20,layout=[49,2], figsize=[12,60])

In [4]:
sorted(list(df.columns))

['BASE_N(CC)(CC)CC',
 'BASE_N=1C(=CC=CC1C)C',
 'BASE_N=1C=CC=CC1',
 'BASE_NC',
 'BASE_NC(C)(C)C',
 'BASE_[K].O=C(O)O',
 'BASE_[K].O=P(O)(O)O',
 'BASE_[K].OC(C)(C)C',
 'BOHR_Br[B:1](Br)Br',
 'BOHR_C1COB([B:1]2OCCO2)O1',
 'BOHR_CC1(C)OB(O[B:1]2OC(C)(C)C(C)(C)O2)OC1(C)C',
 'BOHR_CC1(C)OB([B:1]2OC(C)(C)C(C)(C)O2)OC1(C)C',
 'BOHR_CC1(C)O[BH:1]OC1(C)C',
 'BOHR_CC1CC(C)(C)OB([B:1]2OC(C)CC(C)(C)O2)O1',
 'BOHR_CC[Si](CC)(CC)[B:1]1OC(C)(C)C(C)(C)O1',
 'BOHR_c1ccc2c(c1)O[BH:1]O2',
 'CATALYST_C(#[N+][Pd:1][N+]#CC)C',
 'CATALYST_C=1C=C2[N]3=C(C1)C[P](C(C)C)(C(C)C)[Co+:1]3(C[Si](C)(C)C)[P](C2)(C(C)C)C(C)C',
 'CATALYST_CC(=O)O[Pd:1]OC(=O)C',
 'CATALYST_Cc1cc(C)cc(C)c1[Ir:1](B1OC(C)(C)C(C)(C)O1)(B2OC(C)(C)C(C)(C)O2)B3OC(C)(C)C(C)(C)O3',
 'CATALYST_FC=1C(F)=C(F)C([B:1](C=2C(F)=C(F)C(F)=C(F)C2F)C=3C(F)=C(F)C(F)=C(F)C3F)=C(F)C1F',
 'CATALYST_O1[Si](C)(C)[CH]2=[CH2][Pt+2:1]32(=[C]4N(C=CN4C5CCCCC5)C6CCCCC6)[CH2]=[CH]3[Si]1(C)C',
 'CATALYST_O=C(C(C)(C)C)[O][Co+3:1]12([N]3=C(C[P]2(C(C)C)C(C)C)C=C(C)C=C3C[P]1

In [5]:
#The dataset is heavily imbalanced - 90% of the reactive atoms are not borrylated
#To ensure that our model learns that aliphatic and non-aromatic carbons can also react, we have to do a stratified split
#Since we are doing a train-test split before we featurize the molecules, we have to:
#1. create a bunch of variables for the the reactive center(hybridization, is_aromatic)
#2. Do the stratified kfold
#3. Make sure that the fold ENDS UP IN THE END DATA, while info about the reactive center DOES NOT - THAT IS A TARGET LEAKAGE!!!
def get_if_reactive_aromaticity_hybrid(params_):
    smiles_, mapno_reactive_= params_
    mol = Chem.MolFromSmiles(smiles_)
    idx_reactive = 0
    for atom in mol.GetAtoms():
        if atom.GetAtomMapNum() == mapno_reactive_:
            return atom.GetHybridization(), atom.GetIsAromatic()
    
    atom = mol.GetAtomWithIdx(idx_reactive)
    return atom.GetHybridization(), atom.GetIsAromatic()

In [6]:
list(df.loc[0, 'reactivity_map'].keys())[0]

4

In [7]:
'''for i in range(len(df)):
    d = df.loc[i, 'reactivity_map']
    descending = dict(sorted(d.items(), key=lambda kv: kv[1], reverse=True))
    id_major = list(descending.keys())[0]
    df.loc[i, ['reactive_hybrid', 'reactive_aromatic']] = get_if_reactive_aromaticity_hybrid((df.loc[i, 'substrate_smiles_mapped'], id_major))
    df.loc[i, 'reactive_sp3'] = 0
    for j in range(len(list(descending.keys()))):
        id_ = list(descending.keys())[j]
        hybrid, aromatic = get_if_reactive_aromaticity_hybrid((df.loc[i, 'substrate_smiles_mapped'], id_))
        if hybrid == 4:
            df.loc[i, 'reactive_sp3'] = 1'''
        

"for i in range(len(df)):\n    d = df.loc[i, 'reactivity_map']\n    descending = dict(sorted(d.items(), key=lambda kv: kv[1], reverse=True))\n    id_major = list(descending.keys())[0]\n    df.loc[i, ['reactive_hybrid', 'reactive_aromatic']] = get_if_reactive_aromaticity_hybrid((df.loc[i, 'substrate_smiles_mapped'], id_major))\n    df.loc[i, 'reactive_sp3'] = 0\n    for j in range(len(list(descending.keys()))):\n        id_ = list(descending.keys())[j]\n        hybrid, aromatic = get_if_reactive_aromaticity_hybrid((df.loc[i, 'substrate_smiles_mapped'], id_))\n        if hybrid == 4:\n            df.loc[i, 'reactive_sp3'] = 1"

In [8]:
for i in range(len(df)):
    d = df.loc[i, 'reactivity_map']
    descending = dict(sorted(d.items(), key=lambda kv: kv[1], reverse=True))
    id_major = list(descending.keys())[0]
    df.loc[i, ['reactive_hybrid', 'reactive_aromatic']] = 0
    df.loc[i, 'reactive_sp3'] = 0
    for j in range(len(list(descending.keys()))):
        id_ = list(descending.keys())[j]
        hybrid, aromatic = get_if_reactive_aromaticity_hybrid((df.loc[i, 'substrate_smiles_mapped'], id_))
        if hybrid == 4:
            df.loc[i, 'reactive_sp3'] = 1
        elif hybrid == 3 and aromatic:
            df.loc[i, 'reactive_aromatic'] = 1
        elif hybrid == 3 and not aromatic:
            df.loc[i, 'reactive_hybrid'] = 1

In [9]:
df['hashed'] = 100 * df['reactive_sp3'] + 10 * df['reactive_aromatic'] + df['reactive_hybrid']
df['hashed'] = df['hashed'].astype(int)
df.drop(columns=['reactive_hybrid', 'reactive_sp3','reactive_aromatic'], inplace= True)

In [10]:
df['hashed'].value_counts()

hashed
10     551
100     85
1        1
110      1
Name: count, dtype: int64

In [11]:
df.tail()

,substrate_smiles_mapped,product_smiles_mapped,temperature_degC,reactivity_map,BOHR_CC1(C)OB([B:1]2OC(C)(C)C(C)(C)O2)OC1(C)C,BOHR_CC1(C)O[BH:1]OC1(C)C,BOHR_CC1(C)OB(O[B:1]2OC(C)(C)C(C)(C)O2)OC1(C)C,BOHR_c1ccc2c(c1)O[BH:1]O2,BOHR_CC[Si](CC)(CC)[B:1]1OC(C)(C)C(C)(C)O1,BOHR_C1COB([B:1]2OCCO2)O1,...,OTHER_O(C=1C=C(OC)C=C(OC)C1)C,OTHER_[K].O=S(=O)(O)OOS(=O)(=O)O,OTHER_O1BOC(C)(C)C1(C)C,OTHER_O(C)CCOC,OTHER_ClC(Cl)C(Cl)Cl,OTHER_[Na].O=S(=O)(O)C1=CC=C(C=C1)C,OTHER_[F-][B+3]([F-])([F-])[F-].CC[N+](CC)(CC)CC,reaction_id,new,hashed
633,[cH:1]1[cH:2][cH:3][c:4]2[o:5][cH:6][cH:7][c:8...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,"{6: 0.7446808510638298, 7: 0.2553191489361702}",1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,lit_pub_bo_248,False,10
634,[cH:1]1[cH:2][cH:3][c:4]2[s:5][cH:6][cH:7][c:8...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,"{7: 0.0909090909090909, 6: 0.9090909090909092}",1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,lit_pub_bo_247,False,10
635,[cH:1]1[cH:2][n:3][c:4]2[nH:5][cH:6][cH:7][c:8...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{7: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,lit_pub_bo_76,False,10
636,[cH:1]1[cH:2][n:3][nH:4][cH:5]1,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{1: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,lit_pub_bo_70,False,10
637,[cH:1]1[s:2][cH:3][c:4]2[c:5]1[O:6][CH2:7][CH2...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[s:7][c...,80,{1: 1.0},0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,lit_pub_bo_303,False,10


In [12]:
n_folds = 5
skf = StratifiedKFold(n_splits = n_folds, shuffle=True, random_state=42)
splits = list(skf.split(df.drop(columns=['hashed'], inplace = False), df['hashed']))
train_idx, test_idx = splits[0]

for i in range(n_folds):
    train_idx, val_idx = splits[i]
    df[f'SPLIT_{i+1}'] = ''
    df.loc[train_idx, f'SPLIT_{i+1}'] = 'train'
    df.loc[val_idx,   f'SPLIT_{i+1}'] = 'val'

df.loc[df['new'] == True, ['SPLIT_1', 'SPLIT_2', 'SPLIT_3', 'SPLIT_4', 'SPLIT_5']] = 'new'

/Users/romannemiatyi/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [13]:
df.drop(columns = ['hashed'], inplace = True)

In [14]:
df

,substrate_smiles_mapped,product_smiles_mapped,temperature_degC,reactivity_map,BOHR_CC1(C)OB([B:1]2OC(C)(C)C(C)(C)O2)OC1(C)C,BOHR_CC1(C)O[BH:1]OC1(C)C,BOHR_CC1(C)OB(O[B:1]2OC(C)(C)C(C)(C)O2)OC1(C)C,BOHR_c1ccc2c(c1)O[BH:1]O2,BOHR_CC[Si](CC)(CC)[B:1]1OC(C)(C)C(C)(C)O1,BOHR_C1COB([B:1]2OCCO2)O1,...,OTHER_ClC(Cl)C(Cl)Cl,OTHER_[Na].O=S(=O)(O)C1=CC=C(C=C1)C,OTHER_[F-][B+3]([F-])([F-])[F-].CC[N+](CC)(CC)CC,reaction_id,new,SPLIT_1,SPLIT_2,SPLIT_3,SPLIT_4,SPLIT_5
0,[Br:1][c:2]1[cH:3][cH:4][c:5]([Br:6])[cH:7][cH...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,{4: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,lit_pub_bo_222,False,train,train,train,train,val
1,[Br:1][c:2]1[cH:3][cH:4][c:5]2[nH:6][cH:7][cH:...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{4: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,lit_pub_bo_27,False,val,train,train,train,train
2,[Br:1][c:2]1[cH:3][cH:4][cH:5][c:6]([Br:7])[cH...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,{4: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,lit_pub_bo_206,False,train,val,train,train,train
3,[Br:1][c:2]1[cH:3][cH:4][cH:5][cH:6][c:7]1[Br:8],[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{5: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,lit_pub_bo_207,False,val,train,train,train,train
4,[Br:1][c:2]1[cH:3][cH:4][cH:5][s:6]1,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,{5: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,lit_pub_bo_216,False,train,train,val,train,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
633,[cH:1]1[cH:2][cH:3][c:4]2[o:5][cH:6][cH:7][c:8...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,"{6: 0.7446808510638298, 7: 0.2553191489361702}",1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,lit_pub_bo_248,False,val,train,train,train,train
634,[cH:1]1[cH:2][cH:3][c:4]2[s:5][cH:6][cH:7][c:8...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,"{7: 0.0909090909090909, 6: 0.9090909090909092}",1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,lit_pub_bo_247,False,val,train,train,train,train
635,[cH:1]1[cH:2][n:3][c:4]2[nH:5][cH:6][cH:7][c:8...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{7: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,lit_pub_bo_76,False,train,train,val,train,train
636,[cH:1]1[cH:2][n:3][nH:4][cH:5]1,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{1: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,lit_pub_bo_70,False,train,train,train,val,train


In [15]:
def get_canonical_smiles(smiles_):
    mol = Chem.MolFromSmiles(smiles_)
    mol = Chem.RemoveHs(mol)
    return Chem.MolToSmiles(mol)

df['substrate_smiles_mapped'] = df['substrate_smiles_mapped'].apply(get_canonical_smiles)

In [16]:
cat_smiles_list = []
cat_cols_list = []
for col in list(df.columns).copy():
    if 'CATALYST_' in col:
        cat_smiles_list.append(col.replace('CATALYST_', ''))
        cat_cols_list.append(col)

cat_smiles_df_list = []
for cat_smiles in cat_smiles_list:
    cat_smiles_df_list.append(featurize_catalyst(cat_smiles))

[13:43:26] UFFTYPER: Unrecognized hybridization for atom: 2
[13:43:26] UFFTYPER: Unrecognized atom type: Ir (2)
[13:43:26] UFFTYPER: Unrecognized hybridization for atom: 5
[13:43:26] UFFTYPER: Unrecognized atom type: Ir (5)
[13:43:26] UFFTYPER: Unrecognized hybridization for atom: 2
[13:43:26] UFFTYPER: Unrecognized atom type: Ir (2)
[13:43:26] UFFTYPER: Unrecognized hybridization for atom: 5
[13:43:26] UFFTYPER: Unrecognized atom type: Ir (5)
[13:43:26] UFFTYPER: Unrecognized atom type: Ir6 (9)
[13:43:26] UFFTYPER: Unrecognized atom type: Ir6 (9)
[13:43:26] UFFTYPER: Unrecognized atom type: Ni6+2 (0)
[13:43:27] UFFTYPER: Unrecognized atom type: Ni6+2 (0)
[13:43:27] UFFTYPER: Unrecognized atom type: Rh6 (0)
[13:43:27] UFFTYPER: Unrecognized atom type: Rh6 (16)
[13:43:27] UFFTYPER: Unrecognized atom type: Rh6 (0)
[13:43:27] UFFTYPER: Unrecognized atom type: Rh6 (16)
[13:43:27] UFFTYPER: Unrecognized atom type: Pt6+2 (6)
[13:43:27] UFFTYPER: Unrecognized atom type: Pt6+2 (6)
[13:43:27] U

In [17]:
cat_smiles_df = pd.concat(cat_smiles_df_list)
cat_smiles_df.reset_index(inplace=True, drop=True)
#cat_smiles_df['catalyst_smiles'] = cat_smiles_list
cat_smiles_df.fillna(-99999, inplace=True)
#cat_smiles_df = cat_smiles_df.add_prefix('CATALYST_')
calculated_props = cat_smiles_df.columns

In [18]:
list(cat_smiles_df.columns)

['CATALYST_free_vol_frac_total_6.0A',
 'CATALYST_free_vol_frac_1_above_6.0A',
 'CATALYST_free_vol_frac_1_below_6.0A',
 'CATALYST_free_vol_frac_2_above_6.0A',
 'CATALYST_free_vol_frac_2_below_6.0A',
 'CATALYST_free_vol_frac_3_above_6.0A',
 'CATALYST_free_vol_frac_3_below_6.0A',
 'CATALYST_SASA_within_6.0A',
 'CATALYST_lvl1_H',
 'CATALYST_lvl1_C',
 'CATALYST_lvl1_O',
 'CATALYST_lvl1_N',
 'CATALYST_lvl1_P',
 'CATALYST_lvl1_S',
 'CATALYST_lvl1_F',
 'CATALYST_lvl1_Cl',
 'CATALYST_lvl1_Br',
 'CATALYST_lvl1_I',
 'CATALYST_lvl1_B',
 'CATALYST_lvl1_Si',
 'CATALYST_lvl2_H',
 'CATALYST_lvl2_C',
 'CATALYST_lvl2_O',
 'CATALYST_lvl2_N',
 'CATALYST_lvl2_P',
 'CATALYST_lvl2_S',
 'CATALYST_lvl2_F',
 'CATALYST_lvl2_Cl',
 'CATALYST_lvl2_Br',
 'CATALYST_lvl2_I',
 'CATALYST_lvl2_B',
 'CATALYST_lvl2_Si',
 'CATALYST_lvl3_H',
 'CATALYST_lvl3_C',
 'CATALYST_lvl3_O',
 'CATALYST_lvl3_N',
 'CATALYST_lvl3_P',
 'CATALYST_lvl3_S',
 'CATALYST_lvl3_F',
 'CATALYST_lvl3_Cl',
 'CATALYST_lvl3_Br',
 'CATALYST_lvl3_I',
 'CA

In [19]:
calculated_props

Index(['CATALYST_free_vol_frac_total_6.0A',
       'CATALYST_free_vol_frac_1_above_6.0A',
       'CATALYST_free_vol_frac_1_below_6.0A',
       'CATALYST_free_vol_frac_2_above_6.0A',
       'CATALYST_free_vol_frac_2_below_6.0A',
       'CATALYST_free_vol_frac_3_above_6.0A',
       'CATALYST_free_vol_frac_3_below_6.0A', 'CATALYST_SASA_within_6.0A',
       'CATALYST_lvl1_H', 'CATALYST_lvl1_C', 'CATALYST_lvl1_O',
       'CATALYST_lvl1_N', 'CATALYST_lvl1_P', 'CATALYST_lvl1_S',
       'CATALYST_lvl1_F', 'CATALYST_lvl1_Cl', 'CATALYST_lvl1_Br',
       'CATALYST_lvl1_I', 'CATALYST_lvl1_B', 'CATALYST_lvl1_Si',
       'CATALYST_lvl2_H', 'CATALYST_lvl2_C', 'CATALYST_lvl2_O',
       'CATALYST_lvl2_N', 'CATALYST_lvl2_P', 'CATALYST_lvl2_S',
       'CATALYST_lvl2_F', 'CATALYST_lvl2_Cl', 'CATALYST_lvl2_Br',
       'CATALYST_lvl2_I', 'CATALYST_lvl2_B', 'CATALYST_lvl2_Si',
       'CATALYST_lvl3_H', 'CATALYST_lvl3_C', 'CATALYST_lvl3_O',
       'CATALYST_lvl3_N', 'CATALYST_lvl3_P', 'CATALYST_lvl3_S',
     

In [20]:
df[calculated_props] = df[cat_cols_list].values.dot(cat_smiles_df.values)
df = df.drop(columns=cat_cols_list)
df

,substrate_smiles_mapped,product_smiles_mapped,temperature_degC,reactivity_map,BOHR_CC1(C)OB([B:1]2OC(C)(C)C(C)(C)O2)OC1(C)C,BOHR_CC1(C)O[BH:1]OC1(C)C,BOHR_CC1(C)OB(O[B:1]2OC(C)(C)C(C)(C)O2)OC1(C)C,BOHR_c1ccc2c(c1)O[BH:1]O2,BOHR_CC[Si](CC)(CC)[B:1]1OC(C)(C)C(C)(C)O1,BOHR_C1COB([B:1]2OCCO2)O1,...,CATALYST_lvl2_STD_Hybridization,CATALYST_lvl3_Mean_Partial_Charge,CATALYST_lvl3_Max_Partial_Charge,CATALYST_lvl3_Min_Partial_Charge,CATALYST_lvl3_STD_Partial_Charge,CATALYST_lvl3_Mean_Hybridization,CATALYST_lvl3_Max_Hybridization,CATALYST_lvl3_Min_Hybridization,CATALYST_lvl3_STD_Hybridization,CATALYST_atomic_no
0,[Br:1][c:2]1[cH:3][cH:4][c:5]([Br:6])[cH:7][cH...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,{4: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,1.686548,-99999.000000,-99999.000000,-99999.000000,-99999.000000,4.000000,4.0,4.0,0.000000,77.0
1,[Br:1][c:2]1[cH:3][cH:4][c:5]2[nH:6][cH:7][cH:...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{4: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,1.851640,-99999.000000,-99999.000000,-99999.000000,-99999.000000,4.000000,4.0,4.0,0.000000,77.0
2,[Br:1][c:2]1[cH:3][cH:4][cH:5][c:6]([Br:7])[cH...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,{4: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,1.686548,-99999.000000,-99999.000000,-99999.000000,-99999.000000,4.000000,4.0,4.0,0.000000,77.0
3,[Br:1][c:2]1[cH:3][cH:4][cH:5][cH:6][c:7]1[Br:8],[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{5: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,1.686548,-99999.000000,-99999.000000,-99999.000000,-99999.000000,4.000000,4.0,4.0,0.000000,77.0
4,[Br:1][c:2]1[cH:3][cH:4][cH:5][s:6]1,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,{5: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,1.686548,-99999.000000,-99999.000000,-99999.000000,-99999.000000,4.000000,4.0,4.0,0.000000,77.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
633,[cH:1]1[cH:2][cH:3][c:4]2[o:5][cH:6][cH:7][c:8...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,"{6: 0.7446808510638298, 7: 0.2553191489361702}",1.0,0.0,0.0,0.0,0.0,0.0,...,0.516398,-99999.000000,-99999.000000,-99999.000000,-99999.000000,3.833333,4.0,3.0,0.389249,78.0
634,[cH:1]1[cH:2][cH:3][c:4]2[s:5][cH:6][cH:7][c:8...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,"{7: 0.0909090909090909, 6: 0.9090909090909092}",1.0,0.0,0.0,0.0,0.0,0.0,...,0.516398,-99999.000000,-99999.000000,-99999.000000,-99999.000000,3.833333,4.0,3.0,0.389249,78.0
635,[cH:1]1[cH:2][n:3][c:4]2[nH:5][cH:6][cH:7][c:8...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{7: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,1.686548,-99999.000000,-99999.000000,-99999.000000,-99999.000000,4.000000,4.0,4.0,0.000000,77.0
636,[cH:1]1[cH:2][n:3][nH:4][cH:5]1,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{1: 1.0},1.0,0.0,0.0,0.0,0.0,0.0,...,1.686548,-99999.000000,-99999.000000,-99999.000000,-99999.000000,4.000000,4.0,4.0,0.000000,77.0


In [21]:
bohr_smiles_list = []
bohr_cols_list = []
for col in list(df.columns).copy():
    if 'BOHR_' in col:
        bohr_smiles_list.append(col.replace('BOHR_', ''))
        bohr_cols_list.append(col)

bohr_smiles_df_list = []
for bohr_smiles in bohr_smiles_list:
    print(bohr_smiles)
    bohr_smiles_df_list.append(featurize_borrylator(bohr_smiles))

CC1(C)OB([B:1]2OC(C)(C)C(C)(C)O2)OC1(C)C
CC1(C)O[BH:1]OC1(C)C
CC1(C)OB(O[B:1]2OC(C)(C)C(C)(C)O2)OC1(C)C
c1ccc2c(c1)O[BH:1]O2
CC[Si](CC)(CC)[B:1]1OC(C)(C)C(C)(C)O1
C1COB([B:1]2OCCO2)O1
CC1CC(C)(C)OB([B:1]2OC(C)CC(C)(C)O2)O1
Br[B:1](Br)Br


In [22]:
bohr_smiles_df = pd.concat(bohr_smiles_df_list)
bohr_smiles_df.reset_index(inplace=True, drop=True)
#bohr_smiles_df['bohr_smiles'] = bohr_smiles_list
bohr_smiles_df.fillna(-99999, inplace=True)
#bohr_smiles_df = bohr_smiles_df.add_prefix('BOHR_')
calculated_props = bohr_smiles_df.columns
list(bohr_smiles_df.columns)

['BOHR_free_vol_frac_total_6.0A',
 'BOHR_free_vol_frac_1_above_6.0A',
 'BOHR_free_vol_frac_1_below_6.0A',
 'BOHR_free_vol_frac_2_above_6.0A',
 'BOHR_free_vol_frac_2_below_6.0A',
 'BOHR_free_vol_frac_3_above_6.0A',
 'BOHR_free_vol_frac_3_below_6.0A',
 'BOHR_SASA_within_6.0A',
 'BOHR_lvl1_H',
 'BOHR_lvl1_C',
 'BOHR_lvl1_O',
 'BOHR_lvl1_N',
 'BOHR_lvl1_P',
 'BOHR_lvl1_S',
 'BOHR_lvl1_F',
 'BOHR_lvl1_Cl',
 'BOHR_lvl1_Br',
 'BOHR_lvl1_I',
 'BOHR_lvl1_B',
 'BOHR_lvl1_Si',
 'BOHR_lvl2_H',
 'BOHR_lvl2_C',
 'BOHR_lvl2_O',
 'BOHR_lvl2_N',
 'BOHR_lvl2_P',
 'BOHR_lvl2_S',
 'BOHR_lvl2_F',
 'BOHR_lvl2_Cl',
 'BOHR_lvl2_Br',
 'BOHR_lvl2_I',
 'BOHR_lvl2_B',
 'BOHR_lvl2_Si',
 'BOHR_lvl3_H',
 'BOHR_lvl3_C',
 'BOHR_lvl3_O',
 'BOHR_lvl3_N',
 'BOHR_lvl3_P',
 'BOHR_lvl3_S',
 'BOHR_lvl3_F',
 'BOHR_lvl3_Cl',
 'BOHR_lvl3_Br',
 'BOHR_lvl3_I',
 'BOHR_lvl3_B',
 'BOHR_lvl3_Si',
 'BOHR_lvl0_Mean_Partial_Charge',
 'BOHR_lvl0_Max_Partial_Charge',
 'BOHR_lvl0_Min_Partial_Charge',
 'BOHR_lvl0_STD_Partial_Charge',
 'BOHR

In [23]:
calculated_props

Index(['BOHR_free_vol_frac_total_6.0A', 'BOHR_free_vol_frac_1_above_6.0A',
       'BOHR_free_vol_frac_1_below_6.0A', 'BOHR_free_vol_frac_2_above_6.0A',
       'BOHR_free_vol_frac_2_below_6.0A', 'BOHR_free_vol_frac_3_above_6.0A',
       'BOHR_free_vol_frac_3_below_6.0A', 'BOHR_SASA_within_6.0A',
       'BOHR_lvl1_H', 'BOHR_lvl1_C', 'BOHR_lvl1_O', 'BOHR_lvl1_N',
       'BOHR_lvl1_P', 'BOHR_lvl1_S', 'BOHR_lvl1_F', 'BOHR_lvl1_Cl',
       'BOHR_lvl1_Br', 'BOHR_lvl1_I', 'BOHR_lvl1_B', 'BOHR_lvl1_Si',
       'BOHR_lvl2_H', 'BOHR_lvl2_C', 'BOHR_lvl2_O', 'BOHR_lvl2_N',
       'BOHR_lvl2_P', 'BOHR_lvl2_S', 'BOHR_lvl2_F', 'BOHR_lvl2_Cl',
       'BOHR_lvl2_Br', 'BOHR_lvl2_I', 'BOHR_lvl2_B', 'BOHR_lvl2_Si',
       'BOHR_lvl3_H', 'BOHR_lvl3_C', 'BOHR_lvl3_O', 'BOHR_lvl3_N',
       'BOHR_lvl3_P', 'BOHR_lvl3_S', 'BOHR_lvl3_F', 'BOHR_lvl3_Cl',
       'BOHR_lvl3_Br', 'BOHR_lvl3_I', 'BOHR_lvl3_B', 'BOHR_lvl3_Si',
       'BOHR_lvl0_Mean_Partial_Charge', 'BOHR_lvl0_Max_Partial_Charge',
       'BOHR_lvl0_Mi

In [24]:
df[calculated_props] = df[bohr_cols_list].values.dot(bohr_smiles_df.values)
df = df.drop(columns=bohr_cols_list)
df

/var/folders/xl/v8lcvy855_39wt1534xvs_hw0000gn/T/ipykernel_45403/2010323829.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[calculated_props] = df[bohr_cols_list].values.dot(bohr_smiles_df.values)
/var/folders/xl/v8lcvy855_39wt1534xvs_hw0000gn/T/ipykernel_45403/2010323829.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[calculated_props] = df[bohr_cols_list].values.dot(bohr_smiles_df.values)
/var/folders/xl/v8lcvy855_39wt1534xvs_hw0000gn/T/ipykernel_45403/2010323829.py:1: PerformanceWarning: DataFrame is highly frag

,substrate_smiles_mapped,product_smiles_mapped,temperature_degC,reactivity_map,SOLVENT_CCCCCCC,SOLVENT_N#CC,SOLVENT_ClC=1C=CC=CC1,SOLVENT_CO,SOLVENT_C=1C=CC(=CC1)C,SOLVENT_C1CCCCCCC1,...,BOHR_lvl2_Min_Hybridization,BOHR_lvl2_STD_Hybridization,BOHR_lvl3_Mean_Partial_Charge,BOHR_lvl3_Max_Partial_Charge,BOHR_lvl3_Min_Partial_Charge,BOHR_lvl3_STD_Partial_Charge,BOHR_lvl3_Mean_Hybridization,BOHR_lvl3_Max_Hybridization,BOHR_lvl3_Min_Hybridization,BOHR_lvl3_STD_Hybridization
0,[Br:1][c:2]1[cH:3][cH:4][c:5]([Br:6])[cH:7][cH...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,{4: 1.0},0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,0.012085,0.074641,-0.034832,0.058516,4.0,4.0,4.0,0.0
1,[Br:1][c:2]1[cH:3][cH:4][c:5]2[nH:6][cH:7][cH:...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{4: 1.0},0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,0.012085,0.074641,-0.034832,0.058516,4.0,4.0,4.0,0.0
2,[Br:1][c:2]1[cH:3][cH:4][cH:5][c:6]([Br:7])[cH...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,{4: 1.0},0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,0.012085,0.074641,-0.034832,0.058516,4.0,4.0,4.0,0.0
3,[Br:1][c:2]1[cH:3][cH:4][cH:5][cH:6][c:7]1[Br:8],[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{5: 1.0},0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,0.012085,0.074641,-0.034832,0.058516,4.0,4.0,4.0,0.0
4,[Br:1][c:2]1[cH:3][cH:4][cH:5][s:6]1,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,{5: 1.0},0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,0.012085,0.074641,-0.034832,0.058516,4.0,4.0,4.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
633,[cH:1]1[cH:2][cH:3][c:4]2[o:5][cH:6][cH:7][c:8...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,"{6: 0.7446808510638298, 7: 0.2553191489361702}",1.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,0.012085,0.074641,-0.034832,0.058516,4.0,4.0,4.0,0.0
634,[cH:1]1[cH:2][cH:3][c:4]2[s:5][cH:6][cH:7][c:8...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,100,"{7: 0.0909090909090909, 6: 0.9090909090909092}",1.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,0.012085,0.074641,-0.034832,0.058516,4.0,4.0,4.0,0.0
635,[cH:1]1[cH:2][n:3][c:4]2[nH:5][cH:6][cH:7][c:8...,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{7: 1.0},0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,0.012085,0.074641,-0.034832,0.058516,4.0,4.0,4.0,0.0
636,[cH:1]1[cH:2][n:3][nH:4][cH:5]1,[CH3:1][C:2]1([CH3:3])[O:4][B:5]([c:6]2[cH:7][...,80,{1: 1.0},0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,0.012085,0.074641,-0.034832,0.058516,4.0,4.0,4.0,0.0


In [25]:
substrate_smiles_list = df['substrate_smiles_mapped'].values
extra_rows_list = []

for substrate_smiles in substrate_smiles_list:
    mol_from_smiles = Chem.MolFromSmiles(substrate_smiles)
    descriptor_df = pd.DataFrame([Descriptors.CalcMolDescriptors(mol_from_smiles, missingVal=np.nan)])
    extra_rows_list.append(descriptor_df)

extra_descriptors_df = pd.concat(extra_rows_list)
extra_descriptors_df.reset_index(inplace=True, drop = True)
extra_descriptors_df

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,3.324583,3.324583,1.111759,1.111759,0.648141,9.250000,235.906,231.874,233.867974,42,...,0,0,0,0,0,0,0,0,0,0
1,3.403194,3.403194,1.124630,1.124630,0.666141,10.500000,196.047,189.999,194.968361,50,...,0,0,0,0,0,0,0,0,0,0
2,3.332083,3.332083,1.103588,1.103588,0.648141,9.250000,235.906,231.874,233.867974,42,...,0,0,0,0,0,0,0,0,0,0
3,3.348287,3.348287,1.097222,1.097222,0.648141,9.250000,235.906,231.874,233.867974,42,...,0,0,0,0,0,0,0,0,0,0
4,3.299769,3.299769,1.196759,1.196759,0.550710,8.833333,163.039,160.015,161.913883,32,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
633,5.117870,5.117870,0.956019,0.956019,0.517424,10.222222,118.135,112.087,118.041865,44,...,0,0,0,0,0,0,0,0,0,0
634,2.138889,2.138889,1.349537,1.349537,0.519376,10.222222,134.203,128.155,134.019021,44,...,0,0,0,0,0,0,0,1,0,0
635,4.085648,4.085648,0.956019,0.956019,0.558373,10.222222,118.139,112.091,118.053098,44,...,0,0,0,0,0,0,0,0,0,0
636,3.597222,3.597222,1.694444,1.694444,0.468343,8.000000,68.079,64.047,68.037448,26,...,0,0,0,0,0,0,0,0,0,0


In [26]:
extra_descriptors_df = extra_descriptors_df.add_prefix('DESCRIPTORS_')

In [27]:
df = df.join(extra_descriptors_df)
#df

In [28]:
extra_cols = []
for col in df.columns:
    if ('CATALYST_' in col) or  ('BOHR_' in col) or ('BASE' in col) or ('OTHER' in col) or ('NaK' in col) or ('DESCRIPTORS_' in col) or ('SPLIT_' in col) or ('BOHR_' in col) or ('substrate_smiles_mapped' in col) or ('SOLVENT_' in col) or ('new' in col) or ('LIGAND_' in col) or ('temperature_degC' in col):
        extra_cols.append(col)

extra_df = df[extra_cols].copy()

In [29]:
list(extra_df.columns)

['substrate_smiles_mapped',
 'temperature_degC',
 'SOLVENT_CCCCCCC',
 'SOLVENT_N#CC',
 'SOLVENT_ClC=1C=CC=CC1',
 'SOLVENT_CO',
 'SOLVENT_C=1C=CC(=CC1)C',
 'SOLVENT_C1CCCCCCC1',
 'SOLVENT_C=1C=CC(=CC1)N(C)C',
 'SOLVENT_ClCCCl',
 'SOLVENT_C1CCCCC1',
 'SOLVENT_ClCCl',
 'SOLVENT_N(CC)(C(C)C)C(C)C',
 'SOLVENT_CCCCCC',
 'SOLVENT_O(C)C1CCCC1',
 'SOLVENT_O1CCCC1',
 'SOLVENT_COC(C)(C)C',
 'SOLVENT_ClC=1C=CC=CC1Cl',
 'SOLVENT_C1CCOC1',
 'SOLVENT_ClC(Cl)Cl',
 'SOLVENT_N#N',
 'SOLVENT_C=1C=C(C=CC1C)C',
 'SOLVENT_N(CC)(CC)CC',
 'SOLVENT_O(C)C(C)(C)C',
 'LIGAND_O=S(=O)([O-])CC=1C=NC(=CC1)C2=NC=C(C=C2)C.CCCC[N+](CCCC)(CCCC)CCCC',
 'LIGAND_N=1C=CC(=CC1C=2N=CC=C(C2)C(C)(C)C)C(C)(C)C',
 'LIGAND_O=C(NC1=CC=CC2=C1NC(=C2C)C)C=3C=NC(=CC3)C4=NC=CC=C4',
 'LIGAND_N=1C=C(C(=C2C=CC3=C(N=CC(=C3C)C)C12)C)C',
 'LIGAND_N=1C=C(C=CC1C=2N=CC(=CC2)C)C',
 'LIGAND_O=C(NC(C1=NC(CO1)C(C)C)C(C)CC)C',
 'LIGAND_[Fe].CC(C)P([C]1[CH][CH][CH][CH]1)C(C)C.CC(C)P([C]2[CH][CH][CH][CH]2)C(C)C',
 'LIGAND_O=C(C=CC1=CC=C(C=C1)C(F)(F)F)C=

In [30]:
list(df.columns)

['substrate_smiles_mapped',
 'product_smiles_mapped',
 'temperature_degC',
 'reactivity_map',
 'SOLVENT_CCCCCCC',
 'SOLVENT_N#CC',
 'SOLVENT_ClC=1C=CC=CC1',
 'SOLVENT_CO',
 'SOLVENT_C=1C=CC(=CC1)C',
 'SOLVENT_C1CCCCCCC1',
 'SOLVENT_C=1C=CC(=CC1)N(C)C',
 'SOLVENT_ClCCCl',
 'SOLVENT_C1CCCCC1',
 'SOLVENT_ClCCl',
 'SOLVENT_N(CC)(C(C)C)C(C)C',
 'SOLVENT_CCCCCC',
 'SOLVENT_O(C)C1CCCC1',
 'SOLVENT_O1CCCC1',
 'SOLVENT_COC(C)(C)C',
 'SOLVENT_ClC=1C=CC=CC1Cl',
 'SOLVENT_C1CCOC1',
 'SOLVENT_ClC(Cl)Cl',
 'SOLVENT_N#N',
 'SOLVENT_C=1C=C(C=CC1C)C',
 'SOLVENT_N(CC)(CC)CC',
 'SOLVENT_O(C)C(C)(C)C',
 'LIGAND_O=S(=O)([O-])CC=1C=NC(=CC1)C2=NC=C(C=C2)C.CCCC[N+](CCCC)(CCCC)CCCC',
 'LIGAND_N=1C=CC(=CC1C=2N=CC=C(C2)C(C)(C)C)C(C)(C)C',
 'LIGAND_O=C(NC1=CC=CC2=C1NC(=C2C)C)C=3C=NC(=CC3)C4=NC=CC=C4',
 'LIGAND_N=1C=C(C(=C2C=CC3=C(N=CC(=C3C)C)C12)C)C',
 'LIGAND_N=1C=C(C=CC1C=2N=CC(=CC2)C)C',
 'LIGAND_O=C(NC(C1=NC(CO1)C(C)C)C(C)CC)C',
 'LIGAND_[Fe].CC(C)P([C]1[CH][CH][CH][CH]1)C(C)C.CC(C)P([C]2[CH][CH][CH][CH]2)C(C

In [31]:
mol_df_list = []
for i in list(df.index):
    reaction_id = i
    temp_df = featurize_organic_molecule(df.loc[i, 'substrate_smiles_mapped'], df.loc[i, 'reactivity_map'], extra_df[extra_df.index == i])
    temp_df['reaction_id'] = reaction_id
    mol_df_list.append(temp_df)

SMILES: [Br:1][c:2]1[cH:3][cH:4][c:5]([Br:6])[cH:7][cH:8]1
ranks: [-1. -1.  3.  3. -1. -1.  3.  3.]
classes: {3: [2, 3, 6, 7]}
RMK: dict_keys([4])
AIC: [2 3 6 7]
SMILES: [Br:1][c:2]1[cH:3][cH:4][c:5]2[nH:6][cH:7][cH:8][c:9]2[cH:10]1
ranks: [-1. -1.  3.  6. -1. -1.  2.  4. -1.  5.]
classes: {3: [2], 6: [3], 2: [6], 4: [7], 5: [9]}
RMK: dict_keys([4])
AIC: [2]
AIC: [3]
AIC: [6]
AIC: [7]
AIC: [9]
SMILES: [Br:1][c:2]1[cH:3][cH:4][cH:5][c:6]([Br:7])[cH:8]1
ranks: [-1. -1.  4.  3.  4. -1. -1.  6.]
classes: {4: [2, 4], 3: [3], 6: [7]}
RMK: dict_keys([4])
AIC: [2 4]
AIC: [3]
AIC: [7]
SMILES: [Br:1][c:2]1[cH:3][cH:4][cH:5][cH:6][c:7]1[Br:8]
ranks: [-1. -1.  5.  3.  3.  5. -1. -1.]
classes: {5: [2, 5], 3: [3, 4]}
RMK: dict_keys([5])
AIC: [2 5]
AIC: [3 4]
SMILES: [Br:1][c:2]1[cH:3][cH:4][cH:5][s:6]1
ranks: [-1. -1.  4.  2.  3. -1.]
classes: {4: [2], 2: [3], 3: [4]}
RMK: dict_keys([5])
AIC: [2]
AIC: [3]
AIC: [4]
SMILES: [Br:1][c:2]1[cH:3][n:4][c:5]2[nH:6][cH:7][cH:8][c:9]2[cH:10]1
ranks: [-1. -1. 

In [32]:
per_carbon_df = pd.concat(mol_df_list)
per_carbon_df.reset_index(inplace=True, drop = True)

In [33]:
len(sorted(list(per_carbon_df.columns)))

564

In [34]:
num_only_df = per_carbon_df.select_dtypes(include='number')
non_num_df = per_carbon_df.select_dtypes(exclude='number')

In [35]:
print(f'The number of features before cleaning is: {len(num_only_df.columns)}')

The number of features before cleaning is: 557


In [36]:
from sklearn.feature_selection import VarianceThreshold

variance_filter = VarianceThreshold(threshold=0)
variance_filter.fit(num_only_df)


def remove_correlated_features(df, threshold=0.9):
    corr_matrix = df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    return to_drop

correlated_cols = remove_correlated_features(num_only_df.loc[non_num_df['new'] == False, :])
constant_cols = list(num_only_df.loc[non_num_df['new'] == False, :].columns[~variance_filter.get_support()])
total_drop = constant_cols + correlated_cols

for _ in total_drop:
    if _ not in num_only_df.columns:
        print(_)

num_only_df = num_only_df.drop(total_drop, axis = 1)

/Users/romannemiatyi/anaconda3/lib/python3.13/site-packages/sklearn/feature_selection/_variance_threshold.py:114: RuntimeWarning: Degrees of freedom <= 0 for slice.
  self.variances_ = np.nanvar(X, axis=0)
/Users/romannemiatyi/anaconda3/lib/python3.13/site-packages/sklearn/feature_selection/_variance_threshold.py:122: RuntimeWarning: All-NaN slice encountered
  self.variances_ = np.nanmin(compare_arr, axis=0)


In [37]:
print(f'The number of features after cleaning is: {len(num_only_df.columns)}')

The number of features after cleaning is: 285


In [38]:
sorted(list(per_carbon_df.columns))

['BASE_N(CC)(CC)CC',
 'BASE_N=1C(=CC=CC1C)C',
 'BASE_N=1C=CC=CC1',
 'BASE_NC',
 'BASE_NC(C)(C)C',
 'BASE_[K].O=C(O)O',
 'BASE_[K].O=P(O)(O)O',
 'BASE_[K].OC(C)(C)C',
 'BOHR_SASA_within_6.0A',
 'BOHR_free_vol_frac_1_above_6.0A',
 'BOHR_free_vol_frac_1_below_6.0A',
 'BOHR_free_vol_frac_2_above_6.0A',
 'BOHR_free_vol_frac_2_below_6.0A',
 'BOHR_free_vol_frac_3_above_6.0A',
 'BOHR_free_vol_frac_3_below_6.0A',
 'BOHR_free_vol_frac_total_6.0A',
 'BOHR_lvl0_Max_Hybridization',
 'BOHR_lvl0_Max_Partial_Charge',
 'BOHR_lvl0_Mean_Hybridization',
 'BOHR_lvl0_Mean_Partial_Charge',
 'BOHR_lvl0_Min_Hybridization',
 'BOHR_lvl0_Min_Partial_Charge',
 'BOHR_lvl0_STD_Hybridization',
 'BOHR_lvl0_STD_Partial_Charge',
 'BOHR_lvl1_B',
 'BOHR_lvl1_Br',
 'BOHR_lvl1_C',
 'BOHR_lvl1_Cl',
 'BOHR_lvl1_F',
 'BOHR_lvl1_H',
 'BOHR_lvl1_I',
 'BOHR_lvl1_Max_Hybridization',
 'BOHR_lvl1_Max_Partial_Charge',
 'BOHR_lvl1_Mean_Hybridization',
 'BOHR_lvl1_Mean_Partial_Charge',
 'BOHR_lvl1_Min_Hybridization',
 'BOHR_lvl1_Min_Pa

In [39]:
per_carbon_df

,SUBSTRATE_free_vol_frac_total_6.0A,SUBSTRATE_free_vol_frac_1_above_6.0A,SUBSTRATE_free_vol_frac_1_below_6.0A,SUBSTRATE_free_vol_frac_2_above_6.0A,SUBSTRATE_free_vol_frac_2_below_6.0A,SUBSTRATE_free_vol_frac_3_above_6.0A,SUBSTRATE_free_vol_frac_3_below_6.0A,SUBSTRATE_SASA_within_6.0A,SUBSTRATE_lvl1_H,SUBSTRATE_lvl1_C,...,DESCRIPTORS_fr_tetrazole,DESCRIPTORS_fr_thiazole,DESCRIPTORS_fr_thiocyan,DESCRIPTORS_fr_thiophene,DESCRIPTORS_fr_unbrch_alkane,DESCRIPTORS_fr_urea,reaction_id,SUBSTRATE_lvl4_Max_Hybridization,SUBSTRATE_lvl4_Min_Hybridization,SUBSTRATE_lvl4_STD_Hybridization
0,0.231981,0.051127,0.180854,0.113627,0.118354,0.127363,0.104618,142.107267,1.0,3.0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,0.276681,0.051127,0.225553,0.202808,0.073873,0.172062,0.104618,153.266090,1.0,3.0,...,0,0,0,0,0,0,1,3.0,3.0,0.00000
2,0.280681,0.027373,0.253308,0.042118,0.238562,0.184062,0.096618,170.238112,1.0,3.0,...,0,0,0,0,0,0,1,3.0,3.0,NaN
3,0.244926,0.019373,0.225553,0.019373,0.225553,0.118354,0.126572,133.098504,1.0,1.5,...,0,0,0,0,0,0,1,3.0,3.0,0.00000
4,0.264803,0.027373,0.237431,0.214685,0.050118,0.187722,0.077081,149.498264,1.0,3.0,...,0,0,0,0,0,0,1,4.0,3.0,0.50000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3721,0.138162,0.019373,0.118789,0.058118,0.080044,0.069081,0.069081,99.347715,1.0,3.0,...,0,0,0,0,0,0,636,NaN,NaN,NaN
3722,0.138162,0.019373,0.118789,0.042118,0.096044,0.057299,0.080863,99.347715,1.0,1.5,...,0,0,0,0,0,0,636,NaN,NaN,NaN
3723,0.138162,0.019373,0.118789,0.102789,0.035373,0.072863,0.065299,99.347715,1.0,1.5,...,0,0,0,0,0,0,636,NaN,NaN,NaN
3724,0.245953,0.019373,0.226580,0.122089,0.123863,0.176834,0.069118,156.876830,1.0,1.5,...,0,0,0,1,0,0,637,4.0,3.0,0.57735


In [40]:
per_carbon_df = num_only_df.join(non_num_df)
per_carbon_df

,SUBSTRATE_free_vol_frac_total_6.0A,SUBSTRATE_free_vol_frac_1_above_6.0A,SUBSTRATE_free_vol_frac_1_below_6.0A,SUBSTRATE_free_vol_frac_2_above_6.0A,SUBSTRATE_free_vol_frac_2_below_6.0A,SUBSTRATE_free_vol_frac_3_above_6.0A,SUBSTRATE_free_vol_frac_3_below_6.0A,SUBSTRATE_lvl1_H,SUBSTRATE_lvl1_C,SUBSTRATE_lvl1_O,...,SUBSTRATE_lvl4_Max_Hybridization,SUBSTRATE_lvl4_Min_Hybridization,SUBSTRATE_lvl4_STD_Hybridization,substrate_smiles_mapped,new,SPLIT_1,SPLIT_2,SPLIT_3,SPLIT_4,SPLIT_5
0,0.231981,0.051127,0.180854,0.113627,0.118354,0.127363,0.104618,1.0,3.0,0.0,...,NaN,NaN,NaN,[Br:1][c:2]1[cH:3][cH:4][c:5]([Br:6])[cH:7][cH...,False,train,train,train,train,val
1,0.276681,0.051127,0.225553,0.202808,0.073873,0.172062,0.104618,1.0,3.0,0.0,...,3.0,3.0,0.00000,[Br:1][c:2]1[cH:3][cH:4][c:5]2[nH:6][cH:7][cH:...,False,val,train,train,train,train
2,0.280681,0.027373,0.253308,0.042118,0.238562,0.184062,0.096618,1.0,3.0,0.0,...,3.0,3.0,NaN,[Br:1][c:2]1[cH:3][cH:4][c:5]2[nH:6][cH:7][cH:...,False,val,train,train,train,train
3,0.244926,0.019373,0.225553,0.019373,0.225553,0.118354,0.126572,1.0,1.5,0.0,...,3.0,3.0,0.00000,[Br:1][c:2]1[cH:3][cH:4][c:5]2[nH:6][cH:7][cH:...,False,val,train,train,train,train
4,0.264803,0.027373,0.237431,0.214685,0.050118,0.187722,0.077081,1.0,3.0,0.0,...,4.0,3.0,0.50000,[Br:1][c:2]1[cH:3][cH:4][c:5]2[nH:6][cH:7][cH:...,False,val,train,train,train,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3721,0.138162,0.019373,0.118789,0.058118,0.080044,0.069081,0.069081,1.0,3.0,0.0,...,NaN,NaN,NaN,[cH:1]1[cH:2][n:3][nH:4][cH:5]1,False,train,train,train,val,train
3722,0.138162,0.019373,0.118789,0.042118,0.096044,0.057299,0.080863,1.0,1.5,0.0,...,NaN,NaN,NaN,[cH:1]1[cH:2][n:3][nH:4][cH:5]1,False,train,train,train,val,train
3723,0.138162,0.019373,0.118789,0.102789,0.035373,0.072863,0.065299,1.0,1.5,0.0,...,NaN,NaN,NaN,[cH:1]1[cH:2][n:3][nH:4][cH:5]1,False,train,train,train,val,train
3724,0.245953,0.019373,0.226580,0.122089,0.123863,0.176834,0.069118,1.0,1.5,0.0,...,4.0,3.0,0.57735,[cH:1]1[s:2][cH:3][c:4]2[c:5]1[O:6][CH2:7][CH2...,False,val,train,train,train,train


In [41]:
sorted(list(per_carbon_df.columns))

['BASE_N(CC)(CC)CC',
 'BASE_N=1C=CC=CC1',
 'BASE_NC',
 'BOHR_lvl1_H',
 'BOHR_lvl2_B',
 'BOHR_lvl3_H',
 'BOHR_lvl3_O',
 'CATALYST_free_vol_frac_total_6.0A',
 'CATALYST_lvl0_Mean_Hybridization',
 'CATALYST_lvl1_B',
 'CATALYST_lvl1_C',
 'CATALYST_lvl1_Cl',
 'CATALYST_lvl1_F',
 'CATALYST_lvl1_N',
 'CATALYST_lvl1_O',
 'CATALYST_lvl1_P',
 'CATALYST_lvl1_STD_Partial_Charge',
 'CATALYST_lvl2_C',
 'CATALYST_lvl2_Cl',
 'CATALYST_lvl2_N',
 'CATALYST_lvl2_Si',
 'CATALYST_lvl3_F',
 'CATALYST_lvl3_H',
 'CATALYST_lvl3_O',
 'DESCRIPTORS_AvgIpc',
 'DESCRIPTORS_BCUT2D_CHGHI',
 'DESCRIPTORS_BCUT2D_CHGLO',
 'DESCRIPTORS_BCUT2D_LOGPHI',
 'DESCRIPTORS_BCUT2D_LOGPLOW',
 'DESCRIPTORS_BCUT2D_MRLOW',
 'DESCRIPTORS_BCUT2D_MWHI',
 'DESCRIPTORS_BCUT2D_MWLOW',
 'DESCRIPTORS_BalabanJ',
 'DESCRIPTORS_BertzCT',
 'DESCRIPTORS_EState_VSA2',
 'DESCRIPTORS_EState_VSA3',
 'DESCRIPTORS_EState_VSA4',
 'DESCRIPTORS_EState_VSA5',
 'DESCRIPTORS_EState_VSA6',
 'DESCRIPTORS_EState_VSA7',
 'DESCRIPTORS_EState_VSA8',
 'DESCRIPTORS_

In [42]:
per_carbon_df[per_carbon_df['reaction_id'] == 0]['SUBSTRATE_reactivity']

0    1.0
Name: SUBSTRATE_reactivity, dtype: float64

In [43]:
sorted(list(per_carbon_df.columns))

['BASE_N(CC)(CC)CC',
 'BASE_N=1C=CC=CC1',
 'BASE_NC',
 'BOHR_lvl1_H',
 'BOHR_lvl2_B',
 'BOHR_lvl3_H',
 'BOHR_lvl3_O',
 'CATALYST_free_vol_frac_total_6.0A',
 'CATALYST_lvl0_Mean_Hybridization',
 'CATALYST_lvl1_B',
 'CATALYST_lvl1_C',
 'CATALYST_lvl1_Cl',
 'CATALYST_lvl1_F',
 'CATALYST_lvl1_N',
 'CATALYST_lvl1_O',
 'CATALYST_lvl1_P',
 'CATALYST_lvl1_STD_Partial_Charge',
 'CATALYST_lvl2_C',
 'CATALYST_lvl2_Cl',
 'CATALYST_lvl2_N',
 'CATALYST_lvl2_Si',
 'CATALYST_lvl3_F',
 'CATALYST_lvl3_H',
 'CATALYST_lvl3_O',
 'DESCRIPTORS_AvgIpc',
 'DESCRIPTORS_BCUT2D_CHGHI',
 'DESCRIPTORS_BCUT2D_CHGLO',
 'DESCRIPTORS_BCUT2D_LOGPHI',
 'DESCRIPTORS_BCUT2D_LOGPLOW',
 'DESCRIPTORS_BCUT2D_MRLOW',
 'DESCRIPTORS_BCUT2D_MWHI',
 'DESCRIPTORS_BCUT2D_MWLOW',
 'DESCRIPTORS_BalabanJ',
 'DESCRIPTORS_BertzCT',
 'DESCRIPTORS_EState_VSA2',
 'DESCRIPTORS_EState_VSA3',
 'DESCRIPTORS_EState_VSA4',
 'DESCRIPTORS_EState_VSA5',
 'DESCRIPTORS_EState_VSA6',
 'DESCRIPTORS_EState_VSA7',
 'DESCRIPTORS_EState_VSA8',
 'DESCRIPTORS_

In [44]:
per_carbon_df.to_csv('PER_CARBON_FEATURIZED.csv')

In [45]:
#from sklearn.preprocessing import StandardScaler

In [46]:
smiles = 'CC(N)=O'
mol = Chem.MolFromSmiles(smiles)

df_cp = get_connectivity_patterns(mol, 1)

df_cp = df_cp[[col for col in df_cp.columns if (('1' in col or '2' in col) and 'M' not in col and 'D' not in col) and 'C' in col or 'O' in col or 'N' in col or 'H' in col]]

In [47]:
df_cp.T

,0
lvl1_H,0.000000
lvl1_C,1.000000
lvl1_O,2.000000
lvl1_N,1.000000
lvl1_Cl,0.000000
lvl2_H,5.000000
lvl2_C,0.000000
lvl2_O,0.000000
lvl2_N,0.000000
lvl2_Cl,0.000000
